# ASSET-11: forced-choice option sets, HT-6 sheet, P-FCE

Runtime: CPU. No GPU, no API key, no cost. Installs: none.

Order: **Cell 0** (inputs) → **Cell 1** (self test) → **Cell 2** (stage 1: draft sets and the HT-6 sheet) → one person does **HT-6** → **Cell 3** (stage 2: final files) → **Cell 4** (commit and push) → **Cell 5** (check).

The code is in the repo: `src/fc_options.py`, `scripts/asset11_options.py`, `scripts/asset11_ht6_page.html`. This notebook only calls it.
Nothing here reads a model answer. The sheet made by Cell 2 shows photos of homes beside their gold names: it stays in Drive `human/HT-6/` and on the reviewer's own computer (D202).

In [ ]:
# Cell 0: Drive, repo, and one PASS or FAIL line per input
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/BanglaIndoorVQA'   # the project folder in Drive, as in the other notebooks
REPO = '/content/banglaindoorvqa-main'
OWNER_REPO = 'blkeyd/banglaindoorvqa-main'
TOKEN = userdata.get('GITHUB_TOKEN')
if not TOKEN:
    raise SystemExit('The Colab secret GITHUB_TOKEN is missing, or this notebook has no access to it (key icon on the left).')

def git(*args):
    r = subprocess.run(['git', '-C', REPO, *args], capture_output=True, text=True)
    if r.returncode != 0:
        what = next((a for a in args if not a.startswith('-') and '=' not in a), args[0])
        raise SystemExit('git %s failed:\n%s' % (what, (r.stderr or r.stdout).replace(TOKEN, '***')))
    return r.stdout.strip()

if not os.path.isdir(REPO + '/.git'):
    r = subprocess.run(['git', 'clone', '-q', 'https://%s@github.com/%s.git' % (TOKEN, OWNER_REPO), REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit('git clone failed:\n' + r.stderr.replace(TOKEN, '***'))
git('pull', '-q', '--ff-only')
git('fetch', '-q', '--tags')
print('repo at commit', git('rev-parse', '--short', 'HEAD'))

!python {REPO}/scripts/asset11_options.py inputs --root "{ROOT}"

In [ ]:
# Cell 1: self test on a made-up lexicon (needs no project file). Every line must print OK.
!python {REPO}/scripts/asset11_options.py selftest

In [ ]:
# Cell 2: STAGE 1. Candidates, draft option sets, HT-6 sheet. About 2 to 5 minutes (it reads 492 pictures from Drive).
!python {REPO}/scripts/asset11_options.py build --root "{ROOT}"

## HT-6 happens here

One reviewer, alone, downloads `ht6_review.html` from Drive `human/HT-6/`, follows `HT6_instructions.md`, and uploads `ht6_log.csv` to the same folder. About 1 hour.
Do not run Cell 3 before the complete log is in Drive.

In [ ]:
# Cell 3: STAGE 2. HT-6 log -> final option sets, P-FCE.json, probes/P-FCE.ids.txt, the P-FCE line of conditions.yaml
!python {REPO}/scripts/asset11_options.py apply --root "{ROOT}"

In [ ]:
# Cell 4: commit and push the two repo files that Cell 3 wrote. Run it only after Cell 3 printed STAGE 2 OK.
GIT_NAME, GIT_EMAIL = '', ''    # your Git name and e-mail, as in the other notebooks

print(git('status', '--short'))
if not (GIT_NAME and GIT_EMAIL):
    raise SystemExit('Fill in GIT_NAME and GIT_EMAIL in this cell, then run it again.')
git('add', 'probes/P-FCE.ids.txt', 'configs/conditions.yaml')
git('-c', 'user.name=' + GIT_NAME, '-c', 'user.email=' + GIT_EMAIL, 'commit', '-m',
    'ASSET-11: P-FCE id list and its size in conditions.yaml (D195 to D202)')
git('-c', 'user.name=' + GIT_NAME, '-c', 'user.email=' + GIT_EMAIL, 'pull', '-q', '--rebase')   # in case someone pushed since Cell 0
git('push')
print('pushed commit', git('rev-parse', '--short', 'HEAD'))

In [ ]:
# Cell 5: read the final files from disk again and repeat every check. Run it also before a forced-choice run.
!python {REPO}/scripts/asset11_options.py check --root "{ROOT}"